# Paper 4 — Final Free-GPU Execution

This notebook runs the scientifically matched EviTrust-VQA B0–B5 protocol. It uses a fixed A-OKVQA training subset for calibration and the untouched A-OKVQA validation split for evaluation. Development and synthetic smoke outputs are never promoted as benchmark results.


In [ ]:
import os, sys, json, pathlib, subprocess, random
REPO_URL='https://github.com/junnubabu-ctrl/paper4-selective-kbvqa.git'
ROOT=pathlib.Path('/content/paper4-selective-kbvqa') if pathlib.Path('/content').exists() else pathlib.Path('./paper4-selective-kbvqa')
CAL_N=1000   # fixed calibration subset from A-OKVQA train
EVAL_MAX=None # full A-OKVQA val; set 100 only for a development dry-run
SEED=2026


## 1. Clone and install


In [ ]:
if not (ROOT/'pyproject.toml').exists(): subprocess.run(['git','clone',REPO_URL,str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run(['git','pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[vlm,retrieval,dev]'],check=True)


## 2. GPU and software integrity gate


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a CUDA GPU runtime before continuing.'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,2),'GB')
subprocess.run([sys.executable,'scripts/inspect_environment.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)


## 3. Download official A-OKVQA annotations and COCO validation images


In [ ]:
subprocess.run([sys.executable,'scripts/prepare_data.py','--dataset','aokvqa','--out','datasets'],check=True)
AOK=ROOT/'datasets/aokvqa'; COCO=ROOT/'datasets/coco'; COCO.mkdir(parents=True,exist_ok=True)
valzip=COCO/'val2017.zip'
if not (COCO/'val2017').exists():
    subprocess.run(['wget','-q','https://images.cocodataset.org/zips/val2017.zip','-O',str(valzip)],check=True)
    subprocess.run(['unzip','-q',str(valzip),'-d',str(COCO)],check=True)


## 4. Build train and validation manifests


In [ ]:
MAN=ROOT/'datasets/manifests'; MAN.mkdir(parents=True,exist_ok=True)
train_full=MAN/'aokvqa_train_full.jsonl'; val=MAN/'aokvqa_val.jsonl'
subprocess.run([sys.executable,'scripts/build_manifest.py','aokvqa','--aokvqa-dir',str(AOK),'--coco-dir',str(COCO),'--split','train','--out',str(train_full)],check=True)
subprocess.run([sys.executable,'scripts/build_manifest.py','aokvqa','--aokvqa-dir',str(AOK),'--coco-dir',str(COCO),'--split','val','--out',str(val)],check=True)


## 5. Freeze a calibration subset from A-OKVQA train and download only its images


In [ ]:
rows=[json.loads(x) for x in train_full.read_text().splitlines() if x.strip()]
rng=random.Random(SEED); rng.shuffle(rows); rows=rows[:CAL_N]
cal=MAN/f'aokvqa_train_cal_{CAL_N}_seed{SEED}.jsonl'
cal.write_text('\n'.join(json.dumps(x) for x in rows)+'\n')
subprocess.run([sys.executable,'scripts/download_coco_images.py','--manifest',str(cal),'--out-dir',str(COCO),'--split','train2017'],check=True)


## 6. Run B3 on calibration data


In [ ]:
subprocess.run([sys.executable,'scripts/run_variants.py','--manifest',str(cal),'--variant','B3','--out','results/predictions/aokvqa_cal_B3.jsonl','--checkpoint','results/checkpoints/aokvqa_cal_B3.json'],check=True)
subprocess.run([sys.executable,'scripts/fit_selective_policy.py','--manifest',str(cal),'--predictions','results/predictions/aokvqa_cal_B3.jsonl','--out','results/policies/aokvqa_policy_5pct.json','--target-risk','0.05'],check=True)


## 7. Run matched B0–B3 on untouched A-OKVQA validation data


In [ ]:
for v in ['B0','B1','B2','B3']:
    cmd=[sys.executable,'scripts/run_variants.py','--manifest',str(val),'--variant',v,'--out',f'results/predictions/aokvqa_val_{v}.jsonl','--checkpoint',f'results/checkpoints/aokvqa_val_{v}.json']
    if EVAL_MAX: cmd += ['--max-samples',str(EVAL_MAX)]
    subprocess.run(cmd,check=True)


## 8. Apply frozen calibration and selective policy (B4/B5)


In [ ]:
subprocess.run([sys.executable,'scripts/apply_selective_policy.py','--policy','results/policies/aokvqa_policy_5pct.json','--predictions','results/predictions/aokvqa_val_B3.jsonl','--out','results/predictions/aokvqa_val_B5.jsonl'],check=True)
policy=json.load(open('results/policies/aokvqa_policy_5pct.json'))
threshold=float(policy['threshold']); print('Frozen threshold=',threshold)


## 9. Evaluate B0–B5


In [ ]:
for v in ['B0','B1','B2','B3']:
    subprocess.run([sys.executable,'scripts/evaluate.py','--manifest',str(val),'--predictions',f'results/predictions/aokvqa_val_{v}.jsonl','--dataset','aokvqa','--confidence-field','raw_confidence','--out',f'results/metrics/aokvqa_val_{v}.json'],check=True)
# B4 = calibrated probabilities, no abstention in the evaluator
subprocess.run([sys.executable,'scripts/evaluate.py','--manifest',str(val),'--predictions','results/predictions/aokvqa_val_B5.jsonl','--dataset','aokvqa','--confidence-field','calibrated_confidence','--out','results/metrics/aokvqa_val_B4.json'],check=True)
# B5 = same calibrated probabilities with frozen threshold
subprocess.run([sys.executable,'scripts/evaluate.py','--manifest',str(val),'--predictions','results/predictions/aokvqa_val_B5.jsonl','--dataset','aokvqa','--confidence-field','calibrated_confidence','--threshold',str(threshold),'--out','results/metrics/aokvqa_val_B5.json'],check=True)


## 10. Assemble the manuscript result table


In [ ]:
subprocess.run([sys.executable,'scripts/summarize_variants.py','--b0','results/metrics/aokvqa_val_B0.json','--b1','results/metrics/aokvqa_val_B1.json','--b2','results/metrics/aokvqa_val_B2.json','--b3','results/metrics/aokvqa_val_B3.json','--b4','results/metrics/aokvqa_val_B4.json','--b5','results/metrics/aokvqa_val_B5.json','--out','results/metrics/aokvqa_B0_B5_summary.json'],check=True)


## 11. What remains after this notebook

Run the prespecified source/top-k/verifier/reranker ablations, evidence-corruption experiments, paired statistics, and efficiency logging. Do not manually edit benchmark numbers; manuscript tables should be generated from the archived evaluator JSON files.
